# OOD Main3 Consistency Ablation: Cross-Run Paper Tables

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Any
import importlib

import pandas as pd

try:
    from IPython import get_ipython
    from IPython.display import Markdown, display
except ImportError:  # pragma: no cover
    get_ipython = None
    Markdown = None

    def display(obj: Any) -> None:
        print(obj)


import ood_main3_consistency_precomputed_tables_lib as ood_tables
ood_tables = importlib.reload(ood_tables)


pd.options.display.max_columns = 200


def md(text: str) -> None:
    shell_name = ""
    if get_ipython is not None and get_ipython() is not None:
        shell_name = get_ipython().__class__.__name__
    if Markdown is not None and shell_name == "ZMQInteractiveShell":
        display(Markdown(text))
    else:
        print(text)


inventory_df, summary_df, metrics_df = ood_tables.load_cross_run_bundle_frames()

MAIN_PAPER_HEATMAP_FEATURE_OPTIONS = list(ood_tables.CORE_FEATURE_ORDER)
MAIN_PAPER_HEATMAP_FEATURE_SET = "Attention + PCA final"
MAIN_PAPER_HEATMAP_SAVE_FIGURES = True
MAIN_PAPER_HEATMAP_EXPORT_DIR = (
    Path(
        os.environ.get(
            "OOD_MAIN3_COLLECTION_RESULTS_ROOT",
            str(ood_tables.DEFAULT_RESULTS_COLLECTION_ROOT),
        )
    ).expanduser().resolve()
    / "cross_run_paper_heatmaps"
)


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")


def save_main_paper_heatmap(
    fig,
    *,
    export_dir: Path,
    target_name: str,
    feature_set: str,
) -> Path:
    export_dir.mkdir(parents=True, exist_ok=True)
    feature_slug = slugify(feature_set)
    out_path = export_dir / f"{target_name}__{feature_slug}__main_paper_transfer_heatmap.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    return out_path


## Cross-Run Paper Tables

Standard errors are shown when the underlying rows are available.

- If multiple bundles exist for the same model / feature set, the SE is computed across bundle means.
- Otherwise the SE falls back to the underlying source-environment validation rows or OOD transfer rows.
- In the environment tables, `Validation AUROC` is averaged over the source-validation rows that feed transfer into the listed held-out environment.
- In the per-model transfer heatmaps, diagonal cells use `Validation AUROC` and off-diagonal cells use OOD AUROC.
- Set `MAIN_PAPER_HEATMAP_FEATURE_SET` in the setup cell to choose the shared feature set used in every panel.
- Set `MAIN_PAPER_HEATMAP_SAVE_FIGURES` to control whether the notebook writes PNGs.
- By default, saved heatmaps go to `MAIN_PAPER_HEATMAP_EXPORT_DIR`, which defaults to `OOD_MAIN3_COLLECTION_RESULTS_ROOT` if set, otherwise `ood_tables.DEFAULT_RESULTS_COLLECTION_ROOT / "cross_run_paper_heatmaps"`.
- Each run saves both main-paper target heatmaps when figure export is enabled.


In [ ]:
if inventory_df.empty:
    md(
        "_No populated bundle directories were found. "
        "Set `OOD_MAIN3_PRECOMPUTED_BUNDLE_ROOTS` to one or more saved output directories if needed._"
    )
else:
    for target_name, target_title in ood_tables.target_rows(summary_df, metrics_df):
        md(f"### {target_title}")

        core_table = ood_tables.build_feature_summary_table(
            summary_df,
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.CORE_FEATURE_ORDER,
        )
        md("#### 1. Core Feature Summary")
        display(ood_tables.style_metric_table(core_table))
        core_missing_note = ood_tables.render_missing_feature_note(ood_tables.missing_feature_sets(core_table))
        if core_missing_note is not None:
            md(core_missing_note)

        core_env_table = ood_tables.build_environment_summary_table(
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.CORE_FEATURE_ORDER,
        )
        md("#### 2. Core Feature Summary By Held-Out Environment")
        display(ood_tables.style_metric_table(core_env_table))

        attention_subset_table = ood_tables.build_feature_summary_table(
            summary_df,
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.ATTENTION_SUBSET_FEATURE_ORDER,
        )
        md("#### 3. Attention-Only Subset Summary")
        display(ood_tables.style_metric_table(attention_subset_table))
        attention_missing_note = ood_tables.render_missing_feature_note(ood_tables.missing_feature_sets(attention_subset_table))
        if attention_missing_note is not None:
            md(attention_missing_note)

        attention_env_table = ood_tables.build_environment_summary_table(
            metrics_df,
            target_name=target_name,
            feature_order=ood_tables.ATTENTION_SUBSET_FEATURE_ORDER,
        )
        md("#### 4. Attention-Only Subset Summary By Held-Out Environment")
        display(ood_tables.style_metric_table(attention_env_table))

    main_paper_targets = ood_tables.main_paper_target_rows(summary_df, metrics_df)
    if main_paper_targets:
        md("## Main-Paper OOD Matrices")
        md("Rows are models, columns are feature sets, and each cell is `Mean OOD AUROC +/- SE`.")
        for table_index, (target_name, target_title) in enumerate(main_paper_targets, start=1):
            matrix_df = ood_tables.build_main_paper_ood_matrix(
                summary_df,
                metrics_df,
                target_name=target_name,
                feature_order=ood_tables.CORE_FEATURE_ORDER,
            )
            md(f"### Table {table_index}: {target_title}")
            display(ood_tables.style_text_table(matrix_df))

        shared_heatmap_feature_set = ood_tables.resolve_feature_set_choice(
            MAIN_PAPER_HEATMAP_FEATURE_SET,
            feature_order=ood_tables.CORE_FEATURE_ORDER,
        )
        export_dir = MAIN_PAPER_HEATMAP_EXPORT_DIR.expanduser().resolve()
        if MAIN_PAPER_HEATMAP_SAVE_FIGURES:
            export_dir.mkdir(parents=True, exist_ok=True)

        md("## Main-Paper Transfer Heatmaps")
        md(
            "Each 2x2 figure shows one panel per model, using the same feature set in every panel. "
            "The diagonal uses `Validation AUROC`, off-diagonals use OOD AUROC, "
            "and each annotation is `AUROC +/- SE`."
        )
        md(f"Shared feature set: `{shared_heatmap_feature_set}`")
        md(f"Figure export enabled: `{MAIN_PAPER_HEATMAP_SAVE_FIGURES}`")
        md(f"Figure export directory: `{export_dir}`")
        for figure_index, (target_name, target_title) in enumerate(main_paper_targets, start=1):
            heatmap_fig, selected_df = ood_tables.plot_main_paper_model_heatmaps(
                summary_df,
                metrics_df,
                target_name=target_name,
                target_title=target_title,
                feature_order=ood_tables.CORE_FEATURE_ORDER,
                shared_feature_set=shared_heatmap_feature_set,
                nrows=2,
                ncols=2,
            )
            md(f"### Figure {figure_index}: {target_title}")
            selected_models = [str(row['Model']) for _, row in selected_df.iterrows()]
            if selected_models:
                md("Models shown: " + "; ".join(selected_models) + ".")
            display(heatmap_fig)
            if MAIN_PAPER_HEATMAP_SAVE_FIGURES:
                saved_path = save_main_paper_heatmap(
                    heatmap_fig,
                    export_dir=export_dir,
                    target_name=target_name,
                    feature_set=shared_heatmap_feature_set,
                )
                md(f"Saved figure: `{saved_path}`")
